# 03 - La chaîne ili2ogc sur un transfert réel

**Objectif**
- Enchaîner les commandes sur un transfert réel du dépôt : `fetch-models`, `validate-xtf`, `convert-jsonschema`, `convert-jsonfg`, `convert-sql`, `import-xtf`.
- Savoir ce que produit chaque commande, dans quel format, et ce qu'elle demande en entrée.
- Lire les codes de sortie et les diagnostics.

Jeu de données : distances par rapport à la forêt (canton de Glaris), `../data/source-xtf/ch_np_wal_v1_2.xtf`, avec le modèle dérivé `../collections/waldabstandslinien/views.ili`.

## Vue d'ensemble

```text
                           models.*.ch
                               |  fetch-models (réseau)
                               v
 transfert .xtf  --validate-xtf-->  rapport d'écarts
      |
      +--convert-jsonfg------->  JSON-FG (objets de la VIEW)
      +--import-xtf----------->  INSERT SQL   -> PostGIS
 modèle .ili  --convert-jsonschema--> JSON Schema
              --convert-sql-------->  CREATE TABLE / CREATE VIEW
```

Codes de sortie, communs à toutes les commandes : 0 succès, 2 succès avec remarques (notes, avertissements), 1 échec.

In [ ]:
!mkdir -p modeles sorties

## Étape 1 - fetch-models : télécharger les modèles

Entrée : un ou plusieurs `.ili` ou `.xtf`. Pour un `.ili`, la commande suit les `IMPORTS` ; pour un `.xtf`, elle lit les modèles nommés dans l'en-tête. Sortie : les `.ili` dans le dossier `-o`, à passer ensuite en `--repo`. Un modèle déjà présent est conservé ; `--repo` désigne des modèles locaux à ne pas télécharger (ici, le modèle dérivé).

C'est la seule commande qui accède au réseau.

In [ ]:
!interlis fetch-models ../data/source-xtf/ch_np_wal_v1_2.xtf ../collections/waldabstandslinien/views.ili --repo ../collections/waldabstandslinien -o modeles
!ls modeles

## Étape 2 - validate-xtf : comparer le transfert à son modèle

Entrée : un `.xtf`. Le modèle est trouvé d'après l'en-tête du transfert (avec `--repo`), ou donné par `--model`. La commande contrôle la structure, les types, les énumérations, les références et les contraintes du modèle (`MANDATORY`, `UNIQUE`, ...). Sortie : une ligne par écart, puis un résumé. `--report` écrit en plus un journal SARIF ; `-v` montre aussi les informations.

In [ ]:
!interlis validate-xtf ../data/source-xtf/ch_np_wal_v1_2.xtf --repo modeles; echo "code de sortie : $?"

Résultat : aucune erreur. Les informations masquées (`-v`) sont des contraintes ou des références que la validation ne peut pas évaluer.

### À vous : provoquer une erreur de validation

Écrire dans `sorties/invalide.xtf` une copie du transfert où `inKraft` est remplacé par une valeur d'énumération interdite pour `Rechtsstatus` (les valeurs admises sont `inKraft`, `AenderungMitVorwirkung`, `AenderungOhneVorwirkung`), par exemple `enCours`. Indice : `sed 's/ancien/nouveau/' entrée > sortie`. La cellule suivante valide la copie et attend un échec.

In [ ]:
!cp ../data/source-xtf/ch_np_wal_v1_2.xtf sorties/invalide.xtf

In [ ]:
%%bash
interlis validate-xtf sorties/invalide.xtf --repo modeles > sorties/validation.txt 2>&1
echo "code de sortie : $?"
head -3 sorties/validation.txt

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
messages = open("sorties/validation.txt").read()
assert "not in the enumeration" in messages, "La validation devrait signaler une valeur hors énumération"
print("Exercice réussi : la validation signale", messages.count("[error"), "erreurs.")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
!sed 's/inKraft/enCours/' ../data/source-xtf/ch_np_wal_v1_2.xtf > sorties/invalide.xtf

## Étape 3 - convert-jsonschema : le JSON Schema du modèle

Entrée : un `.ili` (ici le modèle dérivé). Sortie : un JSON Schema (2020-12) avec une définition par classe et par VIEW. Les remarques (`JSONSCHEMA-TYPE-UNSUPPORTED`) signalent des types que la conversion ne sait pas traduire : ils sont notés, pas supprimés en silence.

In [ ]:
!interlis convert-jsonschema ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/schema.json

In [ ]:
import json

schema = json.load(open("sorties/schema.json"))
print("Définitions :", ", ".join(schema["$defs"]))
print("Propriétés de la VIEW linien :", ", ".join(schema["$defs"]["linien"]["properties"]))

`--collection CLASSE` écrit le schéma autonome d'une seule classe ou VIEW, tel que l'API OGC « Schemas » le publie sur `/collections/{id}/schema`.

### À vous : le schéma d'une seule collection

Produire dans `sorties/schema_linien.json` le schéma de la collection `linien` seule, avec l'option `--collection linien` (`interlis convert-jsonschema --help` la décrit). La vérification affiche ensuite les valeurs admises pour la propriété `rechtsstatus`, une énumération.

In [ ]:
!interlis convert-jsonschema ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/schema_linien.json

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
enum = json.load(open("sorties/schema_linien.json"))["properties"]["rechtsstatus"]["enum"]
print("Exercice réussi : valeurs admises", sorted(enum))

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
!interlis convert-jsonschema ../collections/waldabstandslinien/views.ili --repo modeles --collection linien -o sorties/schema_linien.json

## Étape 4 - convert-jsonfg : les données au format JSON-FG

Entrée : un `.xtf` et le modèle (`--model`, ici le modèle dérivé, pour obtenir aussi les objets de la VIEW). Sortie : une `FeatureCollection` JSON-FG.

Dans JSON-FG, la géométrie est dans `place` (dans le système de coordonnées `coordRefSys` de la collection, ici EPSG:2056) ; `geometry` vaut `null` ici. Chaque objet porte son `featureType` : les classes du modèle de base, et les VIEW du modèle dérivé.

In [ ]:
!interlis convert-jsonfg ../data/source-xtf/ch_np_wal_v1_2.xtf --model ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/waldabstandslinien.jsonfg

In [ ]:
from collections import Counter

jsonfg = json.load(open("sorties/waldabstandslinien.jsonfg"))
print("coordRefSys :", jsonfg["coordRefSys"])
print("Objets par featureType :", dict(Counter(f["featureType"] for f in jsonfg["features"])))
exemple = next(f for f in jsonfg["features"] if f["featureType"] == "linien")
exemple["place"]["coordinates"] = exemple["place"]["coordinates"][:2] + ["..."]
print(json.dumps(exemple, indent=2))

### À vous : compter les objets d'une VIEW

Stocker dans `nombre_lignes` le nombre d'objets de `featureType` `linien` du fichier JSON-FG, et dans `identifiants` la liste triée de leurs `id`.

In [ ]:
nombre_lignes = 0
identifiants = []

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
attendu = open("../data/source-xtf/ch_np_wal_v1_2.xtf").read().count("<Waldabstandslinien_V1_2.Geobasisdaten.Waldabstand_Linie ")
assert nombre_lignes == attendu, f"{attendu} lignes dans le transfert, {nombre_lignes} comptées"
assert len(identifiants) == attendu and identifiants == sorted(identifiants)
print("Exercice réussi :", identifiants)

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
lignes = [f for f in jsonfg["features"] if f["featureType"] == "linien"]
nombre_lignes = len(lignes)
identifiants = sorted(f["id"] for f in lignes)

## Étape 5 - convert-sql : le schéma de base de données

Entrée : un `.ili`. Sortie : un script SQL PostgreSQL (ou GeoPackage avec `--dialect gpkg`) : une table par classe, contraintes `UNIQUE`, `FOREIGN KEY`, `CHECK`, et un `CREATE VIEW` par VIEW (notebook 01). Les classes des modèles importés qu'une VIEW utilise sont reprises comme tables, d'où l'option `--repo`.

In [ ]:
!interlis convert-sql ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/schema.sql
!grep -o 'CREATE TABLE "[a-z_]*"' sorties/schema.sql
!sed -n '/CREATE VIEW "linien"/,/;/p' sorties/schema.sql

### À vous : le même schéma pour GeoPackage

Générer le schéma au dialecte GeoPackage dans `sorties/schema_gpkg.sql` (option `--dialect gpkg`).

In [ ]:
!interlis convert-sql ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/schema_gpkg.sql

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
texte = open("sorties/schema_gpkg.sql").read()
assert "gpkg_geometry_columns" in texte, "ce n'est pas le dialecte GeoPackage"
print("Exercice réussi : le script déclare", texte.count("CREATE TABLE"), "tables.")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
!interlis convert-sql ../collections/waldabstandslinien/views.ili --repo modeles --dialect gpkg -o sorties/schema_gpkg.sql

## Étape 6 - import-xtf : les données sous forme de SQL

Entrée : un ou plusieurs `.xtf` (les données, puis les catalogues vers lesquels elles pointent) et le **même** modèle que pour `convert-sql` (`--model`). Sortie : un script `INSERT` dans une transaction, avec la disposition ili2db (jeu de données, BASKET, TID, références par `t_id`).

Le script suppose que le schéma de l'étape 5 existe dans la base. Charger les deux dans PostgreSQL/PostGIS se fait avec `psql -f` ; le chargeur de la pile Docker (`loader/load.py`) exécute les deux scripts avec psycopg2. Ce notebook ne se connecte à aucune base.

In [ ]:
!interlis import-xtf ../data/source-xtf/ch_np_wal_v1_2.xtf --model ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/donnees.sql
!grep -o 'INSERT INTO "*[a-z_]*' sorties/donnees.sql | sort | uniq -c

### À vous : relire le script d'import

Stocker dans `nombre_inserts` le nombre d'instructions `INSERT INTO "waldabstand_linie"` du script `sorties/donnees.sql` (par exemple avec `!grep -c`, ou en Python avec `str.count`).

In [ ]:
nombre_inserts = 0

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
attendu = open("../data/source-xtf/ch_np_wal_v1_2.xtf").read().count("<Waldabstandslinien_V1_2.Geobasisdaten.Waldabstand_Linie ")
assert nombre_inserts == attendu, f"{attendu} objets dans le transfert, {nombre_inserts} INSERT comptés"
print("Exercice réussi.")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
nombre_inserts = open("sorties/donnees.sql").read().count('INSERT INTO "waldabstand_linie"')

## Les attributs obtenus en suivant une référence

Dans la VIEW, `typ_code := W -> WAL -> Code` suit la référence de chaque ligne vers son type (`WAL`) et en lit l'attribut. Depuis ili2ogc 0.9.0, `convert-jsonfg` écrit ces attributs, comme la vue SQL. Les deux cellules suivantes comparent les colonnes de la vue SQL et les propriétés JSON-FG.

In [ ]:
!sed -n '/CREATE VIEW "linien"/,/;/p' sorties/schema.sql | grep -o 'AS "[a-z_]*"'

In [ ]:
print(sorted(exemple["properties"]))

Tous les attributs de la vue SQL sont présents dans les propriétés JSON-FG, à l'exception de la géométrie (dans `place`), des identifiants techniques (`t_id`, `t_ili_tid`, l'`id` JSON-FG en tient lieu) et de trois attributs qui n'ont aucune valeur dans ce transfert (`publiziertbis`, `bemerkungen`, `typ_abkuerzung`) : un attribut sans valeur n'est pas écrit.

La cible de la référence (`WAL`) est ici dans le même transfert. Quand elle est dans un catalogue distinct, il faut le donner.

### Une référence vers un catalogue : `--catalog`

Le transfert des plans directeurs de l'énergie renouvelable (`RichtplanungErneuerbareEnergien_V1_SH.xtf`) pointe vers des énergies décrites dans un catalogue (`RichtplanungErneuerbareEnergien_Katalog.xml`). La VIEW `flaechen` en lit un attribut (`energieform := F -> Energieform -> Reference -> Energieform`). Sans le catalogue, `convert-jsonfg` ne peut pas suivre la référence : il laisse la valeur de côté, le signale par la remarque `JSONFG-VIEW-PATH-UNRESOLVED` et termine avec le code de sortie 2.

In [ ]:
!interlis fetch-models ../collections/richtplanung/views.ili ../data/source-xtf/RichtplanungErneuerbareEnergien_V1_SH.xtf --repo ../collections/richtplanung --repo ../collections/_shared -o modeles
!interlis convert-jsonfg ../data/source-xtf/RichtplanungErneuerbareEnergien_V1_SH.xtf --model ../collections/richtplanung/views.ili --repo modeles --repo ../collections/richtplanung --repo ../collections/_shared -o sorties/richtplanung_sans_catalogue.jsonfg; echo "code de sortie : $?"

Avec `--catalog` (option répétable, un fichier par catalogue), la référence est suivie et la remarque disparaît :

In [ ]:
!interlis convert-jsonfg ../data/source-xtf/RichtplanungErneuerbareEnergien_V1_SH.xtf --catalog ../data/source-xtf/RichtplanungErneuerbareEnergien_Katalog.xml --model ../collections/richtplanung/views.ili --repo modeles --repo ../collections/richtplanung --repo ../collections/_shared -o sorties/richtplanung.jsonfg; echo "code de sortie : $?"

In [ ]:
def proprietes(fichier, type_objet):
    objets = json.load(open(fichier))["features"]
    return sorted(next(f for f in objets if f["featureType"] == type_objet)["properties"])


print("sans catalogue :", proprietes("sorties/richtplanung_sans_catalogue.jsonfg", "flaechen"))
print("avec catalogue :", proprietes("sorties/richtplanung.jsonfg", "flaechen"))
assert "energieform" not in proprietes("sorties/richtplanung_sans_catalogue.jsonfg", "flaechen")
assert "energieform" in proprietes("sorties/richtplanung.jsonfg", "flaechen")

L'attribut `energieform` n'apparaît qu'avec le catalogue. La publication sans base de données (notebook 04) garde donc les attributs obtenus par référence, à condition de passer les catalogues à `convert-jsonfg`.

## Récapitulatif

| Commande | Entrée | Sortie | Réseau |
|---|---|---|---|
| `fetch-models` | `.ili`, `.xtf` | modèles `.ili` locaux | oui |
| `validate-xtf` | `.xtf`, modèle | écarts, journal SARIF | non |
| `convert-jsonschema` | `.ili` | JSON Schema | non |
| `convert-jsonfg` | `.xtf`, modèle | JSON-FG | non |
| `convert-sql` | `.ili` | `CREATE TABLE`, `CREATE VIEW` | non |
| `import-xtf` | `.xtf`, même modèle | `INSERT` | non |
| `convert-sld` | `.ili` de représentation, signatures `.xtf` | SLD | non |

## Suite

Notebook `04_publication.ipynb` : servir ces données avec pygeoapi.